# Hotel Booking Cancellation & Overbooking Decision
## 08 - Cost-Sensitive Threshold and Overbooking Decision

Random Forest was selected as the strongest baseline. This notebook converts cancellation probabilities into a cost-sensitive business decision. Exact monetary costs were not supplied, so all costs below are clearly labelled scenario assumptions.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
RANDOM_STATE=42

## 2. Load Data and Recreate Features

In [ ]:
df=pd.read_csv('../data/processed/hotel_bookings_cleaned.csv')
d=df.copy()
d['total_nights']=d['stays_in_weekend_nights']+d['stays_in_week_nights']
d['total_guests']=d['adults']+d['children']+d['babies']
d['total_previous_bookings']=d['previous_cancellations']+d['previous_bookings_not_canceled']
d['has_special_request']=(d['total_of_special_requests']>0).astype(int)
d['room_type_changed']=(d['reserved_room_type']!=d['assigned_room_type']).astype(int)
d=d.drop(columns=['stays_in_weekend_nights','stays_in_week_nights','adults','children','babies'])
print(d.shape)

## 3. Same Stratified Train/Test Split

In [ ]:
X=d.drop(columns=['is_canceled']); y=d['is_canceled']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=RANDOM_STATE,stratify=y)
num=X_train.select_dtypes(include=np.number).columns.tolist()
cat=X_train.select_dtypes(include=['object','category']).columns.tolist()
print(X_train.shape,X_test.shape)

## 4. Preprocessing and Random Forest

In [ ]:
preprocessor=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),num),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
rf=Pipeline([('preprocessor',preprocessor),('model',RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE,n_jobs=-1))])
rf.fit(X_train,y_train)
prob=rf.predict_proba(X_test)[:,1]
print('Model trained; probabilities generated.')

## 5. Business Meaning of Errors

Positive class = cancellation.

- **False Positive (FP):** predicted cancellation, but guest does not cancel. This can be risky if the hotel overbooks expecting that guest not to arrive.
- **False Negative (FN):** predicted no cancellation, but booking cancels. This can mean a missed opportunity to resell the room.

## 6. Base Cost Scenario

No exact monetary values were supplied. We therefore use **cost units** rather than pretending these are real hotel rupee costs.

Base assumption: FP cost = 2 units and FN cost = 1 unit.

In [ ]:
COST_FP=2
COST_FN=1
print('FP cost:',COST_FP,'FN cost:',COST_FN)

## 7. Search Thresholds 0.01 to 0.99

In [ ]:
rows=[]
for t in np.arange(.01,1,.01):
    pred=(prob>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_test,pred,labels=[0,1]).ravel()
    rows.append({'Threshold':round(t,2),'TN':tn,'FP':fp,'FN':fn,'TP':tp,'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0),'Total_Cost':fp*COST_FP+fn*COST_FN})
threshold_df=pd.DataFrame(rows)
threshold_df.head()

## 8. Minimum-Cost Threshold

In [ ]:
best=threshold_df.loc[threshold_df['Total_Cost'].idxmin()]
print('Best threshold:',best['Threshold'])
print('Minimum cost:',int(best['Total_Cost']))
print('FP:',int(best['FP']),'FN:',int(best['FN']))
print('Precision:',round(best['Precision'],4),'Recall:',round(best['Recall'],4),'F1:',round(best['F1'],4))

## 9. Cost Curve

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(threshold_df['Threshold'],threshold_df['Total_Cost'])
plt.axvline(best['Threshold'],linestyle='--',label=f"Best threshold={best['Threshold']:.2f}")
plt.xlabel('Cancellation Probability Threshold'); plt.ylabel('Total Cost Units'); plt.title('Cost-Sensitive Threshold Selection'); plt.legend(); plt.show()

## 10. Default 0.50 vs Cost-Optimized Threshold

In [ ]:
def summary(t):
    pred=(prob>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_test,pred,labels=[0,1]).ravel()
    return {'Threshold':t,'FP':fp,'FN':fn,'TP':tp,'TN':tn,'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0),'Total_Cost':fp*COST_FP+fn*COST_FN}
pd.DataFrame([summary(.50),summary(float(best['Threshold']))]).round(4)

## 11. Sensitivity Analysis

Because the correct threshold depends on business priorities, test several FP:FN cost ratios.

In [ ]:
scenarios={'Equal Cost 1:1':(1,1),'FP Twice FN 2:1':(2,1),'FP Triple FN 3:1':(3,1),'FN Twice FP 1:2':(1,2)}
out=[]
for name,(cfp,cfn) in scenarios.items():
    best_cost=np.inf; best_t=None; best_fp=best_fn=None
    for t in np.arange(.01,1,.01):
        pred=(prob>=t).astype(int)
        tn,fp,fn,tp=confusion_matrix(y_test,pred,labels=[0,1]).ravel()
        cost=fp*cfp+fn*cfn
        if cost<best_cost: best_cost,best_t,best_fp,best_fn=cost,t,fp,fn
    out.append({'Scenario':name,'FP_Cost':cfp,'FN_Cost':cfn,'Best_Threshold':best_t,'Minimum_Cost':best_cost,'FP':best_fp,'FN':best_fn})
scenario_df=pd.DataFrame(out)
scenario_df.round(2)

## 12. Expected Cancellations

In [ ]:
expected=prob.sum(); actual=y_test.sum()
print('Test bookings:',len(y_test))
print('Expected cancellations:',round(expected,2))
print('Actual cancellations:',int(actual))

### Why sum probabilities?
For probabilities 0.80, 0.70, 0.60, 0.40 and 0.20, the expected number of cancellations is 2.70. This is a group-level expectation, not a claim that exactly 2.7 customers cancel.

## 13. Illustrative Overbooking Policy

In [ ]:
def expected_cancellations(p): return float(np.sum(p))
def conservative_level(p,safety_fraction=.50): return int(np.floor(expected_cancellations(p)*safety_fraction))
print('Expected cancellations:',round(expected_cancellations(prob),2))
print('Illustrative 50% conservative extra-booking level:',conservative_level(prob,.50))

**Important:** The 50% safety fraction is only an illustrative policy example, not an optimized hotel policy. A real deployment needs hotel capacity, room revenue, relocation/compensation cost, cancellation timing and risk tolerance.

## Final Conclusion

The project now moves from classification to decision support:

`Booking data -> Cancellation probability -> Cost-sensitive threshold -> Expected cancellations -> Overbooking decision`

Random Forest supplies cancellation probabilities. FP/FN cost ratios determine which classification threshold is preferable. Summed probabilities estimate expected cancellations, but expected cancellations should not automatically equal extra bookings because actual cancellations are uncertain.